## Iterators and Generators 

when processing huge datasets- such as a 5GB bank transaction ledger or continous smart - meter readings -loading every record into a list at once can exhaust our system's RAM.


Iterators and generators solve this by producing one item at a time on demand (lazy evaluation), keeping memory usage near zero regardless of dtaset size.

## Iterables vs Iterators (iter() and next())

- Iterable : Any container we can loop over (a list, tuple, dict, str, or open file). It holds data, but does not track whichitem we are currently reading.

- Iterator : A stateful cursor created from an iterable. It remembers its exact position and hands us the next item each time we call next(), raising StopIteration when nothing is left.

### Example 1: Warehouse Conveyor Belt Scanner



In [1]:
scanned_parcels = ["PKG-901", "PKG-902", "PKG-903"]

# Create an iterator cursor from the list
bel_cursor = iter(scanned_parcels)

# Pull items one by one on demand

print(next(bel_cursor)) # first item prints
print(next(bel_cursor)) # second item prints
print(next(bel_cursor)) # third item prints

# Calling next() when exhausted raise StopIteration

try:
    print(next(bel_cursor))
except StopIteration:
    print("conveyor belt empty: all parcels processed.")
    

PKG-901
PKG-902
PKG-903
conveyor belt empty: all parcels processed.


how this works:

- iter(scanned_parcels) creates a curosor pointing just before index 0 of scanned_parcels

- Each next(belt_cursor) call advanced the cursor forward by one positiion and returns that single parcel ID

- Once "PKG-903" is returned, the cursor hars reached the end, so the fourth next() call signals completion by raising StopIteration. (This is the exact mechanism Python's for loop uses silently behind the scenes)


## Building a custom Iterator class (__iter__ and __next__)

Any class becomes an iterator if it implements two dunder methods - known together as **Iterator protocol**:

1. __iter__(self): Returns the iterator object itself(return self)

2. __next__(self): Calculates and returns the next value, or raises StopIteration when finished






### Example 2 : Industiral Sensor Heartbeat Monitor

Suppose an IoT Controller needs to poll a factory temperature sensor a fixed no of times (max_cycles), generating a reading ID on each cycle without pre-building a list

In [2]:
class SensorPollIterator:
    def __init__(self, sensor_id: str, max_cycles: int) -> None:
        self.sensor_id = sensor_id
        self.max_cycles = max_cycles
        self.current_cycle = 0

    def __iter__(self) -> "SensorPollIterator":
        return self

    def __next__(self) -> str:
        if self.current_cycle >= self.max_cycles:
            raise StopIteration
        self.current_cycle += 1
        return f"[{self.sensor_id}] Reading #{self.current_cycle}"


# Python's 'for' loop automatically calls __iter__() once, then __next__() reapeatedly
poller = SensorPollIterator("TEMP-BOILER-02", max_cycles=3)

for reading in poller:
    print(reading)

[TEMP-BOILER-02] Reading #1
[TEMP-BOILER-02] Reading #2
[TEMP-BOILER-02] Reading #3


## Generators (yield) - The Pythonic way to create iterators

Writing a full calss with __iter__, __next__ , and manual StopIteration takes alot of boilerplate. A generator function lets us write the exact same lazy iterator using a normal def function and the yield keyword

- return vs yield: return, terminates a function permanatly and destroys its local vaiables, yield pauses the function, hands a value back to the caller, freezes all local variables in memory, and resumes right after the yield line on the next next() call.

### Example 3: Streaming High value Bank Transactions

Imagine scanning a daily ledger of millions of transactins to find transfers greater than 50,000 without loading all matching all records into a list

In [3]:
from typing import Iterator

def stream_large_transfers(transactions: list[tuple[str, float]], min_amount: float = 50000.0) -> Iterator[str]:

    for txn_id,amount in transactions:
        if amount >= min_amount:
            # Pauses here, sends the formatter alert, and waits for the next request
            yield f"ALERT -> {txn_id}: INR {amount:,.2f}"


daily_ledger = [
    ("TXN-101", 1200.0),
    ("TXN-102", 75000.0),
    ("TXN-103", 450.0),
    ("TXN-104", 125000.0)
]

# Calling the function does NOT run the loop yet-it returns a generator object
alert_stream = stream_large_transfers(daily_ledger)
print("Generator object:", alert_stream)

#Items are computed one by one only as we pull them
print("First alert:", next(alert_stream))
for remaining_alert in alert_stream:
    print("Next alert:", remaining_alert)

Generator object: <generator object stream_large_transfers at 0x7071e5cdb3d0>
First alert: ALERT -> TXN-102: INR 75,000.00
Next alert: ALERT -> TXN-104: INR 125,000.00


#### how this code works:
- Calling stream_large_trassfers(daile_ledger) returns a <generator object> immediately without executing a single iteration of the for loop.

- When next(aler_steram) runs, the loop checks "TXN-101" (1200.0, skipped), moves to "TXN-102" (75000, passes) hits yield, and freezes right at TXN-102

- when the for remaining_alert in alert_stream: loop continues, execution unfreezes right after yield, skips TXN-103 , yields TXN-104  and automatically raises StopIteration when daily_ledger ends

## Generators Expressions(...)

just like list comprehensions uses square brackets [...], generator expression use parentheses (...) to create an inline lazy generator without holding the entire result in memory

### Example 4: Power Grid Voltage Spike Monitoring

In [4]:
voltage_readings = [229.5, 231.0, 248.6, 230.2,253.1, 228.9]

# List comprehension: builds the entire list in RAM immediately

spikes_list = [v for v in voltage_readings if v > 240.0]

#Generator expression: computes each spike lazily only when needed

spikes_gen = (v for v in voltage_readings if v > 240.0)

print("List in RAM",spikes_list)
print("Generator object:", spikes_gen)
print("First spike:", next(spikes_gen))
print("Max remaining:", max(spikes_gen))

List in RAM [248.6, 253.1]
Generator object: <generator object <genexpr> at 0x7071e67c0fb0>
First spike: 248.6
Max remaining: 253.1


#### how this works

- (v for v in voltage_readings if v > 240.0) creates a lightweight generator expression that stores only the rule not numbers

- next(spikes_gen) scand forward unitil it finds 248.6 and pauses

- Passing spikes_gen into builtin functions like sum(), max(), or any() consumes the remaninig values one by one without ever creating a list. 

    - Note: Once a generator is consumes , it is empty -looping over spikes_gen a sexond time yields nothing

# Quick Comparison: List vs Generator 

|Feature |List([...]) |Generator(yield/(...))|
|:---|:---|:---|
|Memory Usage     |   Stores all items in RAM at once   |       stores 1 |
|Reusability| Can loop over it multiple times and index (list[0])| single-pass only;exhausted after one full read|
|Best used when|  Dataset is small/medium or we need random indexing/sorting|Dataset is huge, infinite, or streamed line-by-line|




In real software engineering, there are four speicific situations - where a list is either too slow, too memory-heavy , or literally impossible to use.

## Use Case 1: "Stop Early" Searches (Saving Time & CPU, Not Just RAM)

Suppose a payment processor has 5 million card transactions from today, and a fraud analyst just needs to see the first 2 suspicious transactions above 2,000,000 to verify an alert rule.

- why a list is worse: if we write a function that returns a list, Python must scan and process all 5,000,000 records before returning anything - even though we only wanted 2

- why a generator wins: A generator stops computing the instant we get our second match.


In [ ]:
from typing import Iterator

# Simulated 5 million transactins

all_transactions = [ {"id": f"TXN-{i}", "amount": 500.0 if i not in (3,7, 4_999_999) else 250_000.0} for i in range(5_000_000)]

# GENERAOR APPROACH : Check one record at a time and pauses



def find_fraud_stream(txns: list[dict]) -> Iterator[dict]:
    for txn in txns:
        if txn["amount"] > 200_000:
            yield txn

fraud_gen = find_fraud_stream(all_transactions)

# Pull ONLY the first 2 matches-Python never even looks at the remanining 4,999,992 items!
first_match = next(fraud_gen)
second_match = next(fraud_gen)
print("Stopped early with:", first_match, second_match)

scam
scam
scam


#### how this code works:

- all_transactions has matches at index 3, index 7 and index 4,9999,999

- Calling next(fruad_gen) twice runs the for loop only 8 times (indices 0 to 7) and freezes, The other ~4.99 mullion record are never touched, saving seconds of CPU time

## Google note


Yes, a generator saves massive amounts of time when you stop early, because it skips building the rest of the dataset upfront.
While both a list loop and a generator loop stop checking when they hit a break, the time difference happens before the loop even begins.

1. When stopping early (e.g., finding the first match)

Suppose you want to find the first number divisible by 5 (which happens almost immediately, at index 0 or 5):
• The List ([...]): Python must do all the work upfront. It loops 5,000,000 times to build the entire list in RAM before your for loop can check the very first number. Even though your for loop stops early, you already wasted time generating 4,999,995 items you never looked at.
• The Generator ((...)): Python does zero work upfront. It generates the 1st item, checks it, generates the 2nd item, checks it, and stops the moment it finds a match. The remaining 4,999,995 items are never even created.



2. When checking EVERY item (No break)

If your goal requires checking all 5,000,000 items from start to finish without stopping early:
• A generator does not save time. In fact, a list is often slightly faster here because pausing and resuming a generator function 5 million times adds a tiny bit of CPU overhead.

Quick Summary

• Need to stop early / find the first match? Generators are much faster and use almost no RAM.
• Need to process every single item? Generators save RAM, while lists are slightly faster on CPU time (assuming the list fits comfortably in your RAM).





## Use Case 2: Infinite Sequences (where a list is literally impossible)

Suppose an airline ticketing system or hospital lab needs a unique sequential token generator (LAB-1001,LAB-1002,...) thath keeps issuing IDs for as long as the server stays online - whether that is 50 patients today or 500 million over ten years

- why a list is impoosible: we cannot creatte a list of infinite length(while True: list.append(...) will freeze our computer and crash with MemoryError)


- why a generator wins: A generator with while True: and yield runs forever using only a single integer in memory

In [10]:
from typing import Iterator

def lab_toke_generator(prefix: str = "LAB", start: int = 1001) -> Iterator[str]:
    current = start

    while True: #Infinite loop ! Safe ONLY because 'yield' pauses it
        yield f"{prefix}-{current}"
        current += 1

tokens = lab_toke_generator()

#Call next() whenever a new patient arrives-works endlessly with zero RAM growth
print(next(tokens))
print(next(tokens))
print(next(tokens))
tokens = lab_toke_generator()
print()

for i in range(10):
    print(next(tokens))
    i += 1


LAB-1001
LAB-1002
LAB-1003

LAB-1001
LAB-1002
LAB-1003
LAB-1004
LAB-1005
LAB-1006
LAB-1007
LAB-1008
LAB-1009
LAB-1010


### How this code works:

- Normally, while True: locks up a program. Here, yield acts like a handbrake on every single loop cycle.

- current increments by 1 only when next(tokens) is explicitly called, meaning this function can run for months while occupying just a few bytes of RAM

## Use Case 3: Chaining Multi-Step Data Pipelines (Avoiding Temporary Copies)

Imagine a weather station processing raw hourly sensor lines:

1. Steap 1: Read raw strings.

2. Filter our offline "ERR" readings.

3. Convert Celsius strings into Fahrehheit floats


- WHy lists hurt: if each step returns list, Python creates 3 seperate full-sized lists in RAM (raw list -> filterd list -> converted list). On a 2GB dataset,  that easily eats 6 GB+  of memory

- **Why generators win:** Chaining generators creates an **assembly line.** One Single sensor reading travels through step 1-> step 2-> step 3 beforethe next reading is even read


In [1]:
from typing import Iterator

raw_sensor_feed = ["24.5", "ERR", "26.0", "25.2", "ERR", "27.8"]

def valid_readings (feed: list[str]) -> Iterator[str]:
    for item in feed:
        if item != "ERR":
            yield item

def to_fahrenheit (clean_feed: Iterator[str]) -> Iterator[float]:
    for celcius_str in clean_feed:
        yield round((float(celcius_str) * 9/5)+ 32, 1)


# Connect the two generators together - zero intermediate lists created!
step1_clean = valid_readings(raw_sensor_feed)
step2_fahrenheit = to_fahrenheit(step1_clean)

for tempf in step2_fahrenheit:
    print(f"Dashboard display: {tempf}F")


Dashboard display: 76.1F
Dashboard display: 78.8F
Dashboard display: 77.4F
Dashboard display: 82.0F


### How this code works:
 
- Step2_farenheit pulls one item from step1_clean , which pulls one item from raw_sensor_feed

- At no point in time does a list of valid strings or a list of Fahrenheit floats exist in memory - each value flow through the pipeline one by one

## Use Case 4: Proof of Memory Difference (sys.getsizeof)

We can verify the RAM difference directly in Python using sys.getsizeof(), which measures how many bytes an obje t occupies in memory

In [3]:
import sys

# 1 million numbers stored as a list vs as a generator expression

numbers_list = [x*2 for x in range(1_000_000)]
numbers_gen = (x*2 for x in range(1_000_000))

print(f"List size in RAM:   {sys.getsizeof(numbers_list):,} bytes")
print(f"Generator size in RAM: {sys.getsizeof(numbers_gen):,} bytes")


List size in RAM:   8,448,728 bytes
Generator size in RAM: 200 bytes


### How this code works:

- numbes_list - it should all allocate memory slots for all 1,000,000 integers upfront (around 8.4MB)

- numbers_gen only stores the loop state and formula(around 200 bytes) - even if we change 1_000_000 to 1 billion , the geneartor stays 200 bytes while the list would consume 8GB and crash

# Rule of thumb when to pick which ? 

- Use a list when our dataset is small/medium (hundreds of few thousand items), or when we need len() , indexing (item[5]) , sorting(sorted()), or loop through the data multiple times.

- use a generator when reading large files/database rows, building multi step streaming pipelines, searching for the first few matches in a huge dataset, or generating endless IDs'events